# Day 2 — Exploratory Data Analysis
**Source:** `00_Raw_Data_Workbook.xlsx` (synthetic training data)
This notebook documents a reproducible Excel-first workflow. It reads the raw Excel workbook, cleans transaction data, computes KPIs and profiles trends/segments/outliers.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

file_path = Path('00_Raw_Data_Workbook.xlsx')
sales = pd.read_excel(file_path, sheet_name='Raw_Sales')
customers = pd.read_excel(file_path, sheet_name='Customers')
products = pd.read_excel(file_path, sheet_name='Products')
sales.shape, customers.shape, products.shape

((1205, 10), (250, 5), (40, 5))

## 1. Profile data quality
Check row counts, missing values, invalid quantities/discounts, and duplicate business transactions.

In [2]:
sales.isna().sum().to_frame('missing_count')


,missing_count
Order_ID,0
Order_Date,8
Customer_ID,0
Product_ID,0
Region,12
Sales_Channel,0
Quantity,0
Unit_Price,0
Discount_Pct,0
Revenue,0


In [3]:
fingerprint_cols = ['Order_Date','Customer_ID','Product_ID','Region','Sales_Channel','Quantity','Unit_Price','Discount_Pct','Revenue']
sales['_fingerprint'] = sales[fingerprint_cols].astype(str).agg('|'.join, axis=1)
duplicate_count = sales.duplicated('_fingerprint').sum()
invalid_qty_count = (sales['Quantity'] <= 0).sum()
invalid_discount_count = (~sales['Discount_Pct'].between(0, 0.5)).sum()
print({'rows': len(sales), 'duplicate_transactions': int(duplicate_count), 'invalid_quantity': int(invalid_qty_count), 'invalid_discount': int(invalid_discount_count)})

TypeError: sequence item 0: expected str instance, float found

## 2. Clean and standardize
Do not silently guess missing dates. Exclude records with missing dates or invalid quantities/discounts from time-based KPI analysis; label missing regions as Unknown.

In [4]:
sales = sales.drop_duplicates('_fingerprint', keep='first').drop(columns='_fingerprint').copy()
sales['Order_Date'] = pd.to_datetime(sales['Order_Date'], errors='coerce')
sales['Region'] = sales['Region'].fillna('Unknown')
sales.loc[~sales['Discount_Pct'].between(0, 0.5), 'Discount_Pct'] = np.nan
sales.loc[sales['Quantity'] <= 0, 'Quantity'] = np.nan
clean = sales.dropna(subset=['Order_Date', 'Quantity', 'Discount_Pct']).copy()
clean['Gross_Sales'] = clean['Quantity'] * clean['Unit_Price']
clean['Revenue'] = clean['Gross_Sales'] * (1 - clean['Discount_Pct'])
clean['Order_Month'] = clean['Order_Date'].dt.to_period('M').astype(str)
clean.head()

KeyError: Index(['_fingerprint'], dtype='str')

## 3. KPI calculations
Revenue = Quantity × Unit Price × (1 − Discount). Orders = distinct Order_ID. AOV = Revenue / Orders.

In [ ]:
kpis = {
    'Total Revenue': clean['Revenue'].sum(),
    'Total Orders': clean['Order_ID'].nunique(),
    'Total Units': clean['Quantity'].sum(),
    'AOV': clean['Revenue'].sum() / clean['Order_ID'].nunique(),
    'Total Customers': clean['Customer_ID'].nunique(),
}
pd.Series(kpis, name='Value')

## 4. Segment analysis and monthly trend

In [ ]:
monthly = clean.groupby('Order_Month', as_index=False).agg(Revenue=('Revenue','sum'), Orders=('Order_ID','nunique'), Units=('Quantity','sum'))
region_summary = clean.groupby('Region', as_index=False).agg(Revenue=('Revenue','sum'), Orders=('Order_ID','nunique'))
category_summary = clean.merge(products[['Product_ID','Category']], on='Product_ID', how='left').groupby('Category', as_index=False).agg(Revenue=('Revenue','sum'), Orders=('Order_ID','nunique'))
monthly, region_summary, category_summary

## 5. Outlier detection and interpretation
IQR flags unusual revenue values for review; investigate transaction details before labelling them errors. Correlation or segment differences do not establish causation.

In [ ]:
q1, q3 = clean['Revenue'].quantile([0.25, 0.75])
iqr = q3 - q1
outliers = clean[(clean['Revenue'] < q1 - 1.5*iqr) | (clean['Revenue'] > q3 + 1.5*iqr)].sort_values('Revenue', ascending=False)
outliers.head(20)